In [ ]:
!pip install openai-whisper pybind11


In [ ]:
# Compile the C++ BPE tokenizer extension
!export EXT=$(python3 -c "import sysconfig; print(sysconfig.get_config_var('EXT_SUFFIX'))") && \
 export SRC="/kaggle/input/datasets/aneeshshastri/custom-tokenizers/bpe_tokenizer.cpp" && \
 g++ -O3 -Wall -shared -std=c++17 -fPIC $(python3 -m pybind11 --includes) $SRC -o /kaggle/working/bpe_tokenizer$EXT

!ls -l /kaggle/working/ | grep bpe_tokenizer


In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import optax
import pathlib
import librosa
import numpy as np
import math
import whisper
import bpe_tokenizer
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42


In [ ]:
# ── Constants & Configuration ─────────────────────────────────────────────────

NUM_CLASSES = 8
BATCH_SIZE  = 32

RAVDESS_INPUT      = "/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio"
TRANSCRIPTS_OUTPUT = "/kaggle/working/ravdess_transcripts"
CACHE_DIR          = "/kaggle/working/cache"

EMOTION_MAP = {
    '01': 0,  # neutral
    '02': 1,  # calm
    '03': 2,  # happy
    '04': 3,  # sad
    '05': 4,  # angry
    '06': 5,  # fearful
    '07': 6,  # disgust
    '08': 7,  # surprised
}

MEL_CFG = dict(
    sr         = 22050,
    n_fft      = 1024,
    hop_length = 256,       # ~11.6 ms stride at 22 kHz
    n_mels     = 128,
    fmin       = 50,
    fmax       = 8000,
    duration   = 3.0,
)


In [ ]:
# ── Data Parsing (with deduplication) ─────────────────────────────────────────

def parse_ravdess(root: str) -> list[dict]:
    """Parse RAVDESS filenames into records, skipping duplicate stems."""
    records = []
    seen_stems = set()

    for path in sorted(pathlib.Path(root).rglob("*.wav")):
        if path.stem in seen_stems:
            continue
        seen_stems.add(path.stem)

        parts = path.stem.split('-')
        records.append({
            "path":      str(path),
            "label":     EMOTION_MAP[parts[2]],
            "actor":     int(parts[6]),
            "intensity": int(parts[3]),
        })

    print(f"Parsed {len(records)} unique files "
          f"({len(seen_stems)} stems seen, duplicates skipped).")
    return records


In [ ]:
# ── Whisper Transcription (with skip-if-done guard) ───────────────────────────

def transcribe_corpus(records: list[dict], output_path: str,
                      model_size: str = "base") -> str:
    """
    Transcribe audio files from parsed records using Whisper.
    Skips entirely if transcripts already exist on disk.
    """
    out_dir = Path(output_path)

    # Guard: reuse existing transcripts
    if out_dir.exists():
        existing = sorted(out_dir.glob("*.txt"))
        if len(existing) >= len(records):
            print(f"Found {len(existing)} existing transcripts in "
                  f"{output_path}. Skipping Whisper transcription.")
            return " ".join(f.read_text(encoding="utf-8") for f in existing)

    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Transcribing {len(records)} files with Whisper '{model_size}'...")
    model = whisper.load_model(model_size)
    corpus_texts = []

    for rec in tqdm(records, desc="Transcribing RAVDESS"):
        audio_path = Path(rec["path"])
        result = model.transcribe(
            str(audio_path),
            temperature=0.0,
            condition_on_previous_text=False,
            no_speech_threshold=0.6,
        )
        transcription = result["text"].strip()
        (out_dir / f"{audio_path.stem}.txt").write_text(
            transcription, encoding="utf-8"
        )
        corpus_texts.append(transcription)

    return " ".join(corpus_texts)


In [ ]:
# ── Audio Preprocessing ───────────────────────────────────────────────────────

def load_melspec(path: str, cfg: dict = MEL_CFG) -> np.ndarray:
    """Load a WAV file and return a log-mel spectrogram as (H, W, 1)."""
    wav, _ = librosa.load(path, sr=cfg["sr"], mono=True)
    target_len = int(cfg["sr"] * cfg["duration"])

    if len(wav) < target_len:
        wav = np.pad(wav, (0, target_len - len(wav)))
    else:
        wav = wav[:target_len]

    mel = librosa.feature.melspectrogram(
        y=wav, sr=cfg["sr"], n_fft=cfg["n_fft"],
        hop_length=cfg["hop_length"], n_mels=cfg["n_mels"],
        fmin=cfg["fmin"], fmax=cfg["fmax"],
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)          # (n_mels, T)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    return log_mel[..., np.newaxis].astype(np.float32)       # (128, T, 1)


def preprocess_audio_cache(records: list[dict], cache_dir: str):
    """Compute mel spectrograms for all records, caching to disk."""
    cache = pathlib.Path(cache_dir)
    cache.mkdir(exist_ok=True)

    specs, labels = [], []
    for rec in tqdm(records, desc="Caching mel spectrograms"):
        npy_path = cache / (pathlib.Path(rec["path"]).stem + ".npy")
        if not npy_path.exists():
            mel = load_melspec(rec["path"])
            np.save(npy_path, mel)
        else:
            mel = np.load(npy_path)
        specs.append(mel)
        labels.append(rec["label"])

    X = np.stack(specs)
    y = np.array(labels, dtype=np.int32)
    np.save(cache / "X.npy", X)
    np.save(cache / "y.npy", y)
    return X, y


In [ ]:
# ── Text Preprocessing ────────────────────────────────────────────────────────

def preprocess_transcriptions(tokenizer, root_dir: str):
    """
    Load transcript .txt files, BPE-encode them, and pad into
    a static-shaped matrix suitable for XLA / JAX.
    Returns (X_text, max_len, pad_id).
    """
    root_path = Path(root_dir)
    if not root_path.exists() or not root_path.is_dir():
        raise FileNotFoundError(f"Transcript directory not found: {root_dir}")

    txt_files = sorted(root_path.rglob("*.txt"))
    if not txt_files:
        raise ValueError(f"No .txt files found in {root_dir}")

    print(f"Discovered {len(txt_files)} transcript files. Encoding...")
    encoded_sequences = []
    for txt_path in tqdm(txt_files, desc="Encoding Text"):
        text = txt_path.read_text(encoding="utf-8").strip()
        encoded_sequences.append(tokenizer.encode(text))

    max_len = max(len(seq) for seq in encoded_sequences)
    pad_id  = tokenizer.get_vocab_size()   # pad token = vocab_size
    print(f"Max sequence length: {max_len} tokens | PAD_ID: {pad_id}")

    X_text = np.full((len(encoded_sequences), max_len), pad_id, dtype=np.int32)
    for i, seq in enumerate(encoded_sequences):
        X_text[i, :len(seq)] = seq

    return X_text, max_len, pad_id


In [ ]:
# ── Execute Preprocessing Pipeline ────────────────────────────────────────────

# 1. Parse dataset (with dedup)
records = parse_ravdess(RAVDESS_INPUT)

# 2. Transcribe (skips if already done)
full_corpus_string = transcribe_corpus(records, TRANSCRIPTS_OUTPUT, model_size="base")
print(f"Total characters in corpus: {len(full_corpus_string)}")

# 3. Train BPE tokenizer on corpus
Tokenizer = bpe_tokenizer.BPETokenizer()
Tokenizer.train(full_corpus_string, num_merges=1_000)
VOCAB_SIZE = Tokenizer.get_vocab_size()
print(f"Vocab size: {VOCAB_SIZE}")

# 4. Cache mel spectrograms
X_audio, y = preprocess_audio_cache(records, CACHE_DIR)
print(f"X_audio shape: {X_audio.shape} | y shape: {y.shape}")

# 5. Encode transcripts
X_text, MAX_SEQ_LEN, PAD_ID = preprocess_transcriptions(Tokenizer, TRANSCRIPTS_OUTPUT)
print(f"X_text shape: {X_text.shape}")

# Sanity check: audio and text must have the same sample count
assert len(X_audio) == len(X_text) == len(y), (
    f"Mismatch: audio={len(X_audio)}, text={len(X_text)}, labels={len(y)}"
)


In [ ]:
# ── Train / Val / Test Split + VRAM Transfer ──────────────────────────────────

# First split: 80% train+val, 20% test
X_audio_tv, X_audio_test, X_text_tv, X_text_test, y_tv, y_test = train_test_split(
    X_audio, X_text, y, test_size=0.2, random_state=RANDOM_SEED,
)

# Second split: 90% train, 10% val (of the remaining 80%)
X_audio_train, X_audio_val, X_text_train, X_text_val, y_train, y_val = train_test_split(
    X_audio_tv, X_text_tv, y_tv, test_size=0.1, random_state=RANDOM_SEED,
)

# One-way transfer to GPU VRAM
X_audio_train = jax.device_put(X_audio_train)
X_audio_val   = jax.device_put(X_audio_val)
X_audio_test  = jax.device_put(X_audio_test)

X_text_train  = jax.device_put(X_text_train)
X_text_val    = jax.device_put(X_text_val)
X_text_test   = jax.device_put(X_text_test)

y_train = jax.device_put(y_train)
y_val   = jax.device_put(y_val)
y_test  = jax.device_put(y_test)

print(f"Train: audio {X_audio_train.shape}, text {X_text_train.shape}, labels {y_train.shape}")
print(f"Val:   audio {X_audio_val.shape},   text {X_text_val.shape},   labels {y_val.shape}")
print(f"Test:  audio {X_audio_test.shape},  text {X_text_test.shape},  labels {y_test.shape}")


In [ ]:
# ── Batching Utility ──────────────────────────────────────────────────────────

def get_vram_batches(X, y, batch_size=BATCH_SIZE, shuffle=True, seed=42):
    """
    Yield {"input": ..., "labels": ...} batches from VRAM arrays.
    Drops the final incomplete batch for XLA-friendly static shapes.
    """
    num_samples = len(y)
    indices = np.arange(num_samples)

    if shuffle:
        np.random.default_rng(seed).shuffle(indices)

    num_batches = num_samples // batch_size
    for i in range(num_batches):
        idx = indices[i * batch_size : (i + 1) * batch_size]
        yield {"input": X[idx], "labels": y[idx]}


In [ ]:
# ── Model Definitions ─────────────────────────────────────────────────────────


class OptimizedSpecAugment(nnx.Module):
    """Frequency + time masking for mel spectrograms (SpecAugment)."""

    def __init__(self, freq_mask_param: int, time_mask_param: int, rngs: nnx.Rngs):
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param
        # Store the full Rngs object so each forward pass draws a fresh key.
        # (Storing a single key would freeze the augmentation mask.)
        self.rngs = rngs

    def __call__(self, x: jnp.ndarray, training: bool) -> jnp.ndarray:
        if not training:
            return x

        B, T_dim, F_dim = x.shape
        key = self.rngs.augmentation()
        key_f, key_f0, key_t, key_t0 = jax.random.split(key, 4)

        f  = jax.random.randint(key_f,  (B, 1, 1), 0, self.freq_mask_param)
        f0 = jax.random.randint(key_f0, (B, 1, 1), 0, F_dim)
        t  = jax.random.randint(key_t,  (B, 1, 1), 0, self.time_mask_param)
        t0 = jax.random.randint(key_t0, (B, 1, 1), 0, T_dim)

        freq_idx = jnp.arange(F_dim).reshape(1, 1, F_dim)
        time_idx = jnp.arange(T_dim).reshape(1, T_dim, 1)

        freq_mask = (freq_idx < f0) | (freq_idx >= f0 + f)
        time_mask = (time_idx < t0) | (time_idx >= t0 + t)

        x = jnp.where(freq_mask, x, 0.0)
        x = jnp.where(time_mask, x, 0.0)
        return x


# ── 1D Convolutional Block ───────────────────────────────────────────────────

class Conv1DBlock(nnx.Module):
    """1D Conv -> BatchNorm -> GELU -> MaxPool1D"""

    def __init__(self, in_features: int, out_features: int, rngs: nnx.Rngs):
        self.conv = nnx.Conv(
            in_features=in_features, out_features=out_features,
            kernel_size=(3,), padding="SAME", rngs=rngs,
        )
        self.bn = nnx.BatchNorm(num_features=out_features, rngs=rngs)

    def __call__(self, x: jnp.ndarray, training: bool) -> jnp.ndarray:
        x = self.conv(x)
        x = self.bn(x, use_running_average=not training)
        x = jax.nn.gelu(x)
        x = jax.lax.reduce_window(
            x, -jnp.inf, jax.lax.max,
            window_dimensions=(1, 2, 1), window_strides=(1, 2, 1),
            padding="VALID",
        )
        return x


# ── 1D CNN Backbone ──────────────────────────────────────────────────────────

class CNN1DBackbone(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.block1 = Conv1DBlock(in_features=128, out_features=64,  rngs=rngs)
        self.block2 = Conv1DBlock(in_features=64,  out_features=128, rngs=rngs)
        self.block3 = Conv1DBlock(in_features=128, out_features=256, rngs=rngs)

    def __call__(self, x: jnp.ndarray, training: bool) -> jnp.ndarray:
        x = self.block1(x, training)   # (B, 129, 64)
        x = self.block2(x, training)   # (B, 64, 128)
        x = self.block3(x, training)   # (B, 32, 256)
        return x


# ── GRU Cell (kept for future experimentation) ──────────────────────────────
#
# class GRUCell(nnx.Module):
#     def __init__(self, input_size: int, hidden_size: int, rngs: nnx.Rngs):
#         self.Wr = nnx.Linear(input_size + hidden_size, hidden_size, rngs=rngs)
#         self.Wz = nnx.Linear(input_size + hidden_size, hidden_size, rngs=rngs)
#         self.Wh = nnx.Linear(input_size + hidden_size, hidden_size, rngs=rngs)
#         self.hidden_size = hidden_size
#
#     def __call__(self, x: jnp.ndarray, h: jnp.ndarray) -> jnp.ndarray:
#         xh = jnp.concatenate([x, h], axis=-1)
#         r  = jax.nn.sigmoid(self.Wr(xh))
#         z  = jax.nn.sigmoid(self.Wz(xh))
#         xh_reset = jnp.concatenate([x, r * h], axis=-1)
#         h_cand = jnp.tanh(self.Wh(xh_reset))
#         return (1 - z) * h + z * h_cand


# ── Audio Model: 1D-CRNN ────────────────────────────────────────────────────

class EmotionCRNN(nnx.Module):
    """1D CNN Backbone -> Global Max Pool -> Classifier"""

    def __init__(self, num_classes: int = NUM_CLASSES,
                 gru_hidden: int = 128, rngs: nnx.Rngs = None):
        self.spec_augment = OptimizedSpecAugment(
            freq_mask_param=20, time_mask_param=30, rngs=rngs,
        )
        self.cnn     = CNN1DBackbone(rngs)
        self.connect = nnx.Linear(256, gru_hidden, rngs=rngs)
        self.dropout = nnx.Dropout(rate=0.3, rngs=rngs)
        self.fc1     = nnx.Linear(gru_hidden, 64, rngs=rngs)
        self.fc2     = nnx.Linear(64, num_classes, rngs=rngs)
        self.gru_hidden = gru_hidden

    def __call__(self, x: jnp.ndarray, training: bool = True) -> jnp.ndarray:
        # Input: (B, 128, 259, 1)
        x = jnp.squeeze(x, axis=-1)        # (B, 128, 259)
        x = jnp.transpose(x, (0, 2, 1))    # (B, 259, 128)
        x = self.spec_augment(x, training=training)

        x = self.cnn(x, training)           # (B, 32, 256)

        # GRU path (kept commented for future experimentation)
        # B_dim = x.shape[0]
        # h = jnp.zeros((B_dim, self.gru_hidden))
        # def gru_step(h, x_t):
        #     h_new = self.gru(x_t, h)
        #     return h_new, h_new
        # h, _ = jax.lax.scan(gru_step, h, jnp.transpose(x, (1, 0, 2)))

        x = jnp.max(x, axis=1)             # Global max pool -> (B, 256)
        x = jax.nn.gelu(self.connect(x))
        x = self.dropout(x, deterministic=not training)
        x = jax.nn.gelu(self.fc1(x))
        return self.fc2(x)


# ── Text Model: GRU-based RNN ───────────────────────────────────────────────

class TextRNN(nnx.Module):
    def __init__(self, vocab_size: int, pad_id: int,
                 embed_dim: int, hidden_dim: int,
                 num_classes: int, rngs: nnx.Rngs):
        self.pad_id = pad_id
        self.embed  = nnx.Embed(num_embeddings=vocab_size + 1,
                                features=embed_dim, rngs=rngs)
        self.cell   = nnx.GRUCell(in_features=embed_dim,
                                  hidden_features=hidden_dim, rngs=rngs)
        self.rnn    = nnx.RNN(self.cell)
        self.dropout    = nnx.Dropout(rate=0.3, rngs=rngs)
        self.classifier = nnx.Linear(in_features=hidden_dim,
                                     out_features=num_classes, rngs=rngs)

    def __call__(self, x: jax.Array, training: bool = True) -> jax.Array:
        mask = (x != self.pad_id)
        emb  = self.embed(x)
        rnn_out = self.rnn(emb)

        # Masked mean pooling
        mask_exp    = jnp.expand_dims(mask, axis=-1)
        sum_hidden  = jnp.sum(rnn_out * mask_exp, axis=1)
        valid_lens  = jnp.maximum(jnp.sum(mask, axis=1, keepdims=True), 1)
        pooled      = sum_hidden / valid_lens

        x_drop = self.dropout(pooled, deterministic=not training)
        return self.classifier(x_drop)


In [ ]:
# ── Training Utilities ────────────────────────────────────────────────────────


def create_audio_model_and_optimizer(lr: float = 3e-4):
    rngs  = nnx.Rngs(params=0, dropout=1, batch_stats=2, augmentation=3)
    model = EmotionCRNN(num_classes=NUM_CLASSES, gru_hidden=128, rngs=rngs)
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0, peak_value=lr,
        warmup_steps=200, decay_steps=5000, end_value=1e-6,
    )
    opt = nnx.Optimizer(model, optax.adamw(schedule, weight_decay=1e-4), wrt=nnx.Param)
    return model, opt


def create_text_model_and_optimizer(vocab_size: int, pad_id: int, lr: float = 3e-4):
    rngs  = nnx.Rngs(params=0, dropout=1)
    model = TextRNN(
        vocab_size=vocab_size, pad_id=pad_id,
        embed_dim=64, hidden_dim=128,
        num_classes=NUM_CLASSES, rngs=rngs,
    )
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0, peak_value=lr,
        warmup_steps=100, decay_steps=2000, end_value=1e-6,
    )
    opt = nnx.Optimizer(model, optax.adamw(schedule, weight_decay=1e-4), wrt=nnx.Param)
    return model, opt


# ── Unified step functions (work for both audio and text models) ─────────────

@nnx.jit
def train_step(model, optimizer, metrics, batch):
    def loss_fn(model):
        logits = model(batch["input"], training=True)
        loss = optax.softmax_cross_entropy_with_integer_labels(
            logits, batch["labels"]
        ).mean()
        return loss, logits

    (loss, logits), grads = nnx.value_and_grad(loss_fn, has_aux=True)(model)
    optimizer.update(model, grads)
    metrics.update(loss=loss, logits=logits, labels=batch["labels"])
    return loss


@nnx.jit
def eval_step(model, metrics, batch):
    logits = model(batch["input"], training=False)
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits, batch["labels"]
    ).mean()
    metrics.update(loss=loss, logits=logits, labels=batch["labels"])
    return loss


# ── Test Set Evaluation (works for any model) ────────────────────────────────

def evaluate_test_set(model, X_test, y_test, batch_size=BATCH_SIZE):
    """
    Evaluate on the held-out test set with its own batching logic
    (handles the final partial batch via math.ceil).
    """
    test_metrics = nnx.MultiMetric(
        loss=nnx.metrics.Average('loss'),
        accuracy=nnx.metrics.Accuracy(),
    )
    model.eval()

    num_samples = len(y_test)
    num_batches = math.ceil(num_samples / batch_size)
    print(f"Evaluating {num_samples} test samples across {num_batches} batches...")

    for i in range(num_batches):
        start = i * batch_size
        end   = min((i + 1) * batch_size, num_samples)
        batch = {"input": X_test[start:end], "labels": y_test[start:end]}
        eval_step(model, test_metrics, batch)

    results = test_metrics.compute()
    print("-" * 40)
    print(f"  Loss:     {results['loss']:.4f}")
    print(f"  Accuracy: {results['accuracy']:.4f}")
    print("-" * 40)
    return results


In [ ]:
# ── Audio Model Training ──────────────────────────────────────────────────────

AUDIO_EPOCHS = 100

audio_model, audio_optimizer = create_audio_model_and_optimizer()

metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average('loss'),
    accuracy=nnx.metrics.Accuracy(),
)
val_metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average('loss'),
    accuracy=nnx.metrics.Accuracy(),
)

print("Training EmotionCRNN (audio)...")
for epoch in range(AUDIO_EPOCHS):
    audio_model.train()
    for batch in get_vram_batches(X_audio_train, y_train, shuffle=True, seed=epoch):
        train_step(audio_model, audio_optimizer, metrics, batch)

    audio_model.eval()
    for batch in get_vram_batches(X_audio_val, y_val, shuffle=False):
        eval_step(audio_model, val_metrics, batch)

    print(f"Epoch {epoch:3d} | Train: {metrics.compute()} | Val: {val_metrics.compute()}")
    metrics.reset()
    val_metrics.reset()


In [ ]:
# ── Text Model Training ───────────────────────────────────────────────────────

TEXT_EPOCHS = 50

txt_model, txt_optimizer = create_text_model_and_optimizer(
    vocab_size=VOCAB_SIZE, pad_id=PAD_ID,
)

metrics.reset()
val_metrics.reset()

print("Training TextRNN...")
for epoch in range(TEXT_EPOCHS):
    txt_model.train()
    for batch in get_vram_batches(X_text_train, y_train, shuffle=True, seed=epoch):
        train_step(txt_model, txt_optimizer, metrics, batch)

    txt_model.eval()
    for batch in get_vram_batches(X_text_val, y_val, shuffle=False):
        eval_step(txt_model, val_metrics, batch)

    print(f"Epoch {epoch:3d} | Train: {metrics.compute()} | Val: {val_metrics.compute()}")
    metrics.reset()
    val_metrics.reset()


In [ ]:
# ── Test Set Evaluation (Both Models) ─────────────────────────────────────────

print("=" * 40)
print("EmotionCRNN (Audio) — Test Results")
print("=" * 40)
evaluate_test_set(audio_model, X_audio_test, y_test)

print()

print("=" * 40)
print("TextRNN — Test Results")
print("=" * 40)
evaluate_test_set(txt_model, X_text_test, y_test)


In [ ]:
# ── Leave-One-Subject-Out (LOSO) Final Evaluation ────────────────────────────
#
# For each of the 24 RAVDESS actors, we:
#   1. Hold out that actor as the test set.
#   2. Stratified-random-split the remaining actors into train / val.
#   3. Train both models from scratch.
#   4. Record per-actor test accuracy.
#
# The final reported score is the mean accuracy across all 24 folds.

import copy

actors     = np.array([r["actor"] for r in records])
unique_ids = np.unique(actors)
print(f"LOSO evaluation over {len(unique_ids)} actors: {unique_ids.tolist()}\n")

per_actor_acc_audio = {}
per_actor_acc_text  = {}

for fold, held_out in enumerate(unique_ids):
    print(f"\n{'='*60}")
    print(f"  LOSO Fold {fold+1}/{len(unique_ids)}  —  held-out actor {held_out}")
    print(f"{'='*60}")

    # ── 1. Split by actor ────────────────────────────────────────────────────
    test_mask  = (actors == held_out)
    train_mask = ~test_mask

    X_audio_fold_test = X_audio[test_mask]
    X_text_fold_test  = X_text[test_mask]
    y_fold_test       = y[test_mask]

    X_audio_pool = X_audio[train_mask]
    X_text_pool  = X_text[train_mask]
    y_pool       = y[train_mask]

    # ── 2. Stratified train / val split (90 / 10) ────────────────────────────
    (X_audio_fold_train, X_audio_fold_val,
     X_text_fold_train,  X_text_fold_val,
     y_fold_train,       y_fold_val) = train_test_split(
        X_audio_pool, X_text_pool, y_pool,
        test_size=0.1, stratify=y_pool, random_state=RANDOM_SEED,
    )

    # Move to GPU
    X_audio_fold_train = jax.device_put(X_audio_fold_train)
    X_audio_fold_val   = jax.device_put(X_audio_fold_val)
    X_audio_fold_test  = jax.device_put(X_audio_fold_test)
    X_text_fold_train  = jax.device_put(X_text_fold_train)
    X_text_fold_val    = jax.device_put(X_text_fold_val)
    X_text_fold_test   = jax.device_put(X_text_fold_test)
    y_fold_train       = jax.device_put(y_fold_train)
    y_fold_val         = jax.device_put(y_fold_val)
    y_fold_test        = jax.device_put(y_fold_test)

    print(f"  train {y_fold_train.shape[0]} | val {y_fold_val.shape[0]} "
          f"| test {y_fold_test.shape[0]}")

    # ── 3a. Train fresh audio model ──────────────────────────────────────────
    fold_audio_model, fold_audio_opt = create_audio_model_and_optimizer()

    fold_metrics = nnx.MultiMetric(
        loss=nnx.metrics.Average('loss'),
        accuracy=nnx.metrics.Accuracy(),
    )
    fold_val_metrics = nnx.MultiMetric(
        loss=nnx.metrics.Average('loss'),
        accuracy=nnx.metrics.Accuracy(),
    )

    for epoch in range(AUDIO_EPOCHS):
        fold_audio_model.train()
        for batch in get_vram_batches(X_audio_fold_train, y_fold_train,
                                      shuffle=True, seed=epoch):
            train_step(fold_audio_model, fold_audio_opt, fold_metrics, batch)

        fold_audio_model.eval()
        for batch in get_vram_batches(X_audio_fold_val, y_fold_val,
                                      shuffle=False):
            eval_step(fold_audio_model, fold_val_metrics, batch)

        if (epoch + 1) % 25 == 0 or epoch == 0:
            tc = fold_metrics.compute()
            vc = fold_val_metrics.compute()
            print(f"  [Audio] Epoch {epoch+1:3d} | "
                  f"Train loss {tc['loss']:.4f} acc {tc['accuracy']:.4f} | "
                  f"Val loss {vc['loss']:.4f} acc {vc['accuracy']:.4f}")
        fold_metrics.reset()
        fold_val_metrics.reset()

    audio_results = evaluate_test_set(fold_audio_model,
                                      X_audio_fold_test, y_fold_test)
    per_actor_acc_audio[int(held_out)] = float(audio_results["accuracy"])

    # ── 3b. Train fresh text model ───────────────────────────────────────────
    fold_txt_model, fold_txt_opt = create_text_model_and_optimizer(
        vocab_size=VOCAB_SIZE, pad_id=PAD_ID,
    )
    fold_metrics.reset()
    fold_val_metrics.reset()

    for epoch in range(TEXT_EPOCHS):
        fold_txt_model.train()
        for batch in get_vram_batches(X_text_fold_train, y_fold_train,
                                      shuffle=True, seed=epoch):
            train_step(fold_txt_model, fold_txt_opt, fold_metrics, batch)

        fold_txt_model.eval()
        for batch in get_vram_batches(X_text_fold_val, y_fold_val,
                                      shuffle=False):
            eval_step(fold_txt_model, fold_val_metrics, batch)

        if (epoch + 1) % 25 == 0 or epoch == 0:
            tc = fold_metrics.compute()
            vc = fold_val_metrics.compute()
            print(f"  [Text]  Epoch {epoch+1:3d} | "
                  f"Train loss {tc['loss']:.4f} acc {tc['accuracy']:.4f} | "
                  f"Val loss {vc['loss']:.4f} acc {vc['accuracy']:.4f}")
        fold_metrics.reset()
        fold_val_metrics.reset()

    text_results = evaluate_test_set(fold_txt_model,
                                     X_text_fold_test, y_fold_test)
    per_actor_acc_text[int(held_out)] = float(text_results["accuracy"])

# ── Summary ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  LOSO Results — Per-Actor Accuracy")
print("=" * 60)
print(f"{'Actor':>7s}  {'Audio':>8s}  {'Text':>8s}")
print("-" * 30)
for actor_id in sorted(per_actor_acc_audio):
    print(f"{actor_id:>7d}  "
          f"{per_actor_acc_audio[actor_id]:>8.4f}  "
          f"{per_actor_acc_text[actor_id]:>8.4f}")

mean_audio = np.mean(list(per_actor_acc_audio.values()))
mean_text  = np.mean(list(per_actor_acc_text.values()))
print("-" * 30)
print(f"{'Mean':>7s}  {mean_audio:>8.4f}  {mean_text:>8.4f}")
print("=" * 60)